In [6]:
import pandas as pd
import json
import os

os.makedirs('processed', exist_ok=True)

# 1. Load both files
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

# 2. Check duplicates and nulls
print(f"Ledger Nulls:\n{ledger.isnull().sum()}")
print(f"Gateway Duplicates: {gateway.duplicated().sum()}\n")

# 3-4. Reconciliation Logic
# Merging on transaction_id
recon = pd.merge(ledger, gateway, on='transaction_id', how='outer', suffixes=('_ledger', '_gateway'))

# 5. Identify Amount Mismatches
missing_in_gateway = recon[recon['amount_usd_gateway'].isna()]
missing_in_ledger = recon[recon['amount_usd_ledger'].isna()]

amount_mismatches = recon[
    (recon['amount_usd_ledger'].notna()) &
    (recon['amount_usd_gateway'].notna()) &
    (recon['amount_usd_ledger'] != recon['amount_usd_gateway'])
]

# 6. Identify Status Mismatches
status_mismatches = recon[
    (recon['status_ledger'].notna()) &
    (recon['status_gateway'].notna()) &
    (recon['status_ledger'] != recon['status_gateway'])
]

# 7. Build final reconciliation report
recon['is_reconciled'] = (recon['amount_usd_ledger'] == recon['amount_usd_gateway']) & \
                         (recon['status_ledger'] == recon['status_gateway'])

# 8. Generate summary metrics
metrics = {
    "total_ledger_count": int(len(ledger)),
    "total_gateway_count": int(len(gateway)),
    "missing_in_gateway": int(len(missing_in_gateway)),
    "missing_in_ledger": int(len(missing_in_ledger)),
    "amount_mismatches": int(len(amount_mismatches)),
    "status_mismatches": int(len(status_mismatches)),
    "fully_reconciled": int(recon['is_reconciled'].sum())
}

with open('summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("--- ALL FILES GENERATED ---")
print(json.dumps(metrics, indent=2))

Ledger Nulls:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway Duplicates: 0

--- ALL FILES GENERATED ---
{
  "total_ledger_count": 10,
  "total_gateway_count": 9,
  "missing_in_gateway": 2,
  "missing_in_ledger": 1,
  "amount_mismatches": 2,
  "status_mismatches": 1,
  "fully_reconciled": 5
}


In [5]:
files_to_download = [
    ('missing_in_gateway.csv', missing_in_gateway),
    ('missing_in_ledger.csv', missing_in_ledger),
    ('amount_mismatches.csv', amount_mismatches),
    ('status_mismatches.csv', status_mismatches),
    ('reconciliation_report.csv', recon)
]

for filename, df in files_to_download:
    df.to_csv(filename, index=False)
    print(f"Created: {filename}")


with open('summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)
print("Created: summary_metrics.json")


from google.colab import files
for filename, _ in files_to_download:
    files.download(filename)
files.download('summary_metrics.json')

Created: missing_in_gateway.csv
Created: missing_in_ledger.csv
Created: amount_mismatches.csv
Created: status_mismatches.csv
Created: reconciliation_report.csv
Created: summary_metrics.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>